In [1]:
# This script converts .wav files to 224px grayscale mel spectrograms for inference.
# It needs to be separate from the inference script because it uses librosa, which
# is not available on the nice GPU environment the MA416 instructors give us.
# TODO make an environment that has both librosa and torch with GPU support.

samples_path = "Samples/"
resolution = 224  # Spectrogram resolution

# Import necessary libraries
import librosa
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from tqdm import tqdm


# Define function to generate mel spectrogram
def generate_mel_spectrogram(audio_path, output_path, resolution=224):
    """
    Generate a square Mel spectrogram image from an audio file.
    
    Args:
        audio_path: Path to input .wav file
        output_path: Path to save output .png file
        resolution: Image resolution (width and height in pixels)
    
    Returns:
        tuple: (audio_duration, pixel_values_array)
    """
    # Load audio file
    y, sr = librosa.load(audio_path, sr=None)
    
    # Compute Mel spectrogram
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=resolution)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Create figure without axes/borders
    dpi = 100
    fig_size = resolution / dpi
    fig = plt.figure(figsize=(fig_size, fig_size), dpi=dpi)
    ax = plt.Axes(fig, [0., 0., 1., 1.])
    ax.set_axis_off()
    fig.add_axes(ax)
    
    # Plot spectrogram
    # origin lower is so we have low frequencies at the bottom
    ax.imshow(mel_spec_db, aspect='auto', origin='lower', cmap='gray')
    
    # Save image
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0, dpi=dpi)
    plt.close(fig)

    
    # Calculate audio duration
    duration = librosa.get_duration(y=y, sr=sr)
    
    return duration

In [2]:
# Actually go through the samples path and convert them
input_path = Path(samples_path)
output_path = input_path # I'm lazy

# Find all .wav files
wav_files = sorted(list(input_path.glob('*.wav')))
if not wav_files:
    print(f"No .wav files found in {samples_path}")
    exit()
print(f"Found {len(wav_files)} .wav files")
print(f"Generating grayscale mel spectrograms of resolution {resolution}x{resolution}px")

for wav_file in tqdm(wav_files, desc="Processing"):
    # Generate output filename
    image_filename = wav_file.stem + '.png'
    output_file = output_path / image_filename
    duration = generate_mel_spectrogram(wav_file, output_file, resolution)

Found 10 .wav files
Generating grayscale mel spectrograms of resolution 224x224px


Processing: 100%|██████████| 10/10 [00:03<00:00,  2.94it/s]
